# DreamSense: Automated Dream Interpretation and Sentiment Prediction


## Abstract

DreamSense is an intelligent system designed to interpret dreams and predict whether the underlying sentiment is positive or negative. Leveraging natural language processing and machine learning techniques, this project aims to provide insightful analysis of dream narratives, assisting users in understanding the emotional inclination of their dreams.

---

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
import joblib

## Data Loading

In this section, the dataset containing dream narratives and their corresponding sentiment labels is loaded for further analysis and model development. The data will be explored and preprocessed to ensure quality and suitability for the dream interpretation and sentiment prediction tasks.

In [2]:
# ======================
# 📂 2. Load the Dataset
# ======================
df = pd.read_csv("data/rsos_cleaned.csv").dropna(subset=["text_dream"])
df = df[df['NegativeEmotions'].isin([0, 1])].reset_index(drop=True)

print("Shape:", df.shape)
df['label'] = df['NegativeEmotions'].apply(lambda x: 'negative' if x == 1 else 'positive')

Shape: (10733, 7)


## 3. Clean & Preprocess Text

To prepare the dream narratives for analysis, the text data is cleaned and preprocessed. This involves converting all text to lowercase, removing special characters and punctuation, and normalizing whitespace. The cleaned text is stored in a new column for subsequent processing steps.

In [3]:
# ==============================
# ✨ 3. Clean & Preprocess Text
# ==============================
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text_dream"].apply(clean_text)

## 4. Vectorize Text (TF-IDF)

The cleaned dream texts are transformed into numerical feature vectors using the TF-IDF (Term Frequency-Inverse Document Frequency) method. This process converts the text into a format suitable for machine learning by capturing the importance of words and word pairs (bigrams), while removing common English stop words. The resulting feature matrix is used as input for model training.

In [4]:
# ===============================
# 🧠 4. Vectorize Text (TF-IDF)
# ===============================
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words="english")
X = vectorizer.fit_transform(df["clean_text"])
y = df["label"]


## 5. Train/Test Split

The dataset is split into training and testing sets to evaluate model performance. Stratified sampling ensures that both sets maintain the same proportion of positive and negative labels.

## Scale TF-IDF Vectors

TF-IDF feature vectors are scaled using standardization to improve model performance and ensure consistent feature ranges. Scaling is performed without centering due to the sparse nature of the data.

In [5]:
# ==========================
# 🎲 5. Train/Test Split
# ==========================
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
# ============================
# 📊 Scale TF-IDF Vectors
# ============================
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Train and Evaluate Classifiers

Multiple machine learning classifiers are trained and evaluated to predict the sentiment of dream narratives. The models include Logistic Regression, Random Forest, and a Multi-Layer Perceptron (MLP) neural network. Each model's performance is assessed using classification metrics and ROC AUC scores to determine their effectiveness in distinguishing between positive and negative dreams.

In [6]:
# =====================================
# 🤖 6. Train and Evaluate Classifiers
# =====================================

from sklearn.neural_network import MLPClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42)
}

for name, model in models.items():
    print(f"\n🔍 Training: {name}")
    if "MLP" in name:
        model.fit(X_train_scaled, y_train.values.ravel())
        y_pred = model.predict(X_test_scaled)
        proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train.values.ravel())
        y_pred = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
    print(classification_report(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, proba))


🔍 Training: Logistic Regression
              precision    recall  f1-score   support

    negative       0.82      0.71      0.76      1001
    positive       0.78      0.87      0.82      1146

    accuracy                           0.79      2147
   macro avg       0.80      0.79      0.79      2147
weighted avg       0.80      0.79      0.79      2147

ROC AUC: 0.8716292433569921

🔍 Training: Random Forest
              precision    recall  f1-score   support

    negative       0.84      0.87      0.86      1001
    positive       0.89      0.85      0.87      1146

    accuracy                           0.86      2147
   macro avg       0.86      0.86      0.86      2147
weighted avg       0.86      0.86      0.86      2147

ROC AUC: 0.9263254197809171

🔍 Training: MLP (Neural Net)
              precision    recall  f1-score   support

    negative       0.71      0.73      0.72      1001
    positive       0.76      0.74      0.75      1146

    accuracy                        

## 7. Hyperparameter Tuning

To optimize model performance, hyperparameter tuning is performed on the Random Forest classifier using grid search with cross-validation. The search explores different combinations of tree count, maximum depth, and minimum samples required to split a node. The best parameters are selected based on ROC AUC scores.

In [7]:
# =============================
# 🧪 7. Hyperparameter Tuning
# =============================
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}

gs = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, 
                  cv=StratifiedKFold(n_splits=3), scoring='roc_auc', n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print("\n✅ Best Parameters:", gs.best_params_)

Fitting 3 folds for each of 36 candidates, totalling 108 fits

✅ Best Parameters: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 300}


## 8. Train and Evaluate Best Classifier

The best Random Forest model, identified through hyperparameter tuning, is retrained and evaluated on the test set. Performance metrics, including the classification report and ROC AUC score, are presented to assess the model's effectiveness in predicting dream sentiment.

In [8]:
# =====================================
# 🤖 7. Train and Evaluate Best Classifier
# =====================================
model = gs.best_estimator_
y_pred = model.predict(X_test)

print("\n🧾 Classification Report:")
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))


🧾 Classification Report:
              precision    recall  f1-score   support

    negative       0.85      0.88      0.86      1001
    positive       0.89      0.86      0.87      1146

    accuracy                           0.87      2147
   macro avg       0.87      0.87      0.87      2147
weighted avg       0.87      0.87      0.87      2147

ROC AUC: 0.9300812625419954


In [9]:
# =========================
# 💾 8. Save Best Model
# =========================
joblib.dump(gs.best_estimator_, "models/dream_model.pkl")
joblib.dump(vectorizer, "models/vectorizer.pkl")
print("\nModel + Vectorizer saved.")


Model + Vectorizer saved.


## 📝 Summary

In this project, we developed an end-to-end pipeline for automated dream interpretation and sentiment prediction. Starting from data loading and text preprocessing, we transformed dream narratives into numerical features using TF-IDF vectorization. Multiple machine learning models were trained and evaluated, with hyperparameter tuning applied to optimize performance. The best classifier demonstrated strong predictive capability in distinguishing between positive and negative dreams. This workflow provides a robust foundation for further exploration and enhancement of automated dream analysis systems.